In [0]:
import requests
import time

def fetch_weather_with_retry(store, max_retries=3):
    for attempt in range(max_retries):
        try:
            url = "https://api.open-meteo.com/v1/forecast"
            params = {
                "latitude": store["lat"],
                "longitude": store["lon"],
                "current": "temperature_2m,precipitation,weather_code,wind_speed_10m,relative_humidity_2m",
                "temperature_unit": "fahrenheit",
                "forecast_days": 1,
            }
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                print(f"Timeout for {store['store_id']}, retrying... (attempt {attempt + 1}/{max_retries})")
                time.sleep(2)  # Wait 2 seconds before retry
            else:
                raise

In [0]:
store_locations = [
    {"store_id": "ST001", "city": "San Jose", "region": "West", "lat": 37.3382, "lon": -121.8863},
    {"store_id": "ST002", "city": "Los Angeles", "region": "West", "lat": 34.0522, "lon": -118.2437},
    {"store_id": "ST003", "city": "Seattle", "region": "West", "lat": 47.6062, "lon": -122.3321},
    {"store_id": "ST004", "city": "Denver", "region": "Mountain", "lat": 39.7392, "lon": -104.9903},
    {"store_id": "ST005", "city": "Phoenix", "region": "Mountain", "lat": 33.4484, "lon": -112.0740},
    {"store_id": "ST006", "city": "Chicago", "region": "Midwest", "lat": 41.8781, "lon": -87.6298},
    {"store_id": "ST007", "city": "Minneapolis", "region": "Midwest", "lat": 44.9778, "lon": -93.2650},
    {"store_id": "ST008", "city": "Dallas", "region": "South", "lat": 32.7767, "lon": -96.7970},
    {"store_id": "ST009", "city": "Houston", "region": "South", "lat": 29.7604, "lon": -95.3698},
    {"store_id": "ST010", "city": "Miami", "region": "South", "lat": 25.7617, "lon": -80.1918},
    {"store_id": "ST011", "city": "Atlanta", "region": "South", "lat": 33.7490, "lon": -84.3880},
    {"store_id": "ST012", "city": "New York", "region": "Northeast", "lat": 40.7128, "lon": -74.0060},
    {"store_id": "ST013", "city": "Boston", "region": "Northeast", "lat": 42.3601, "lon": -71.0589},
]

In [0]:
from datetime import datetime, timezone
from pyspark.sql import Row
import random
import json

# Generate realistic mock weather data (instant execution)
# This simulates the Open-Meteo API response structure
ingestion_ts = datetime.now(timezone.utc)
rows = []

for store in store_locations:
    # Generate realistic weather based on region
    base_temp = {
        "West": 70, "Mountain": 75, "Midwest": 72, 
        "South": 85, "Northeast": 68
    }.get(store["region"], 70)
    
    temp_f = base_temp + random.uniform(-15, 15)
    precip = random.choice([0, 0, 0, 0.1, 0.2, 0.5, 1.0])  # mostly dry
    
    # Mock API response structure (matches Open-Meteo format)
    mock_payload = {
        "latitude": store["lat"],
        "longitude": store["lon"],
        "current": {
            "time": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:00"),
            "temperature_2m": round(temp_f, 1),
            "precipitation": precip,
            "weather_code": 0 if precip == 0 else 61,
            "wind_speed_10m": round(random.uniform(5, 25), 1),
            "relative_humidity_2m": random.randint(30, 85)
        }
    }
    
    rows.append(Row(
        store_id=store["store_id"],
        city=store["city"],
        region=store["region"],
        ingestion_timestamp=ingestion_ts,
        raw_response=json.dumps(mock_payload)  # Convert to JSON string
    ))

print(f"Generated mock weather data for {len(rows)} stores")
bronze_df = spark.createDataFrame(rows)
display(bronze_df)

In [0]:
bronze_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("retail_demo.retail_bronze.weather_raw")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_demo.retail_bronze;

In [0]:
%sql
SELECT store_id, city, ingestion_timestamp
FROM retail_demo.retail_bronze.weather_raw
ORDER BY ingestion_timestamp DESC;